In [ ]:
import json
import pandas as pd

# 1. Get secure key
secure_key = dbutils.secrets.get(scope="azure-storage", key="blob-key")

# 2. Read JSON using Spark with authentication inline
# Use multiLine option to handle different JSON formats
# Point to the base directory - Spark will automatically discover partitions
file_path = "wasbs://xp-project-ecommece@xpdataproject.blob.core.windows.net/kafka-data/ecommerce.brz_brands/"

# Try with multiLine=true first (for pretty-printed JSON or JSON arrays)
try:
    df_temp = (spark.read
        .format("json")
        .option("fs.azure.account.key.xpdataproject.blob.core.windows.net", secure_key)
        .option("multiLine", "true")
        .load(file_path)
    )
    
    # Force evaluation and check if we have valid columns
    schema = df_temp.schema
    
    # If we only have _corrupt_record, the format is wrong
    if len(schema.fields) == 1 and schema.fields[0].name == "_corrupt_record":
        raise Exception("Corrupt records detected, trying different format")
    
    df = df_temp
    print(f"Successfully parsed with multiLine=true")
    
except Exception as e1:
    print(f"multiLine=true failed: {str(e1)[:100]}")
    # Try with multiLine=false (newline-delimited JSON)
    try:
        df = (spark.read
            .format("json")
            .option("fs.azure.account.key.xpdataproject.blob.core.windows.net", secure_key)
            .option("multiLine", "false")
            .load(file_path)
        )
        print(f"Successfully parsed with multiLine=false (JSONL format)")
    except Exception as e2:
        print(f"Both JSON parsing attempts failed")
        print(f"Attempting manual read and inspection...")
        
        # Manual inspection - read raw content
        try:
            # Use text format to read raw content
            raw_df = (spark.read
                .format("text")
                .option("fs.azure.account.key.xpdataproject.blob.core.windows.net", secure_key)
                .load(file_path)
            )
            raw_lines = [row.value for row in raw_df.limit(5).collect()]
            print("\nFirst 5 lines of file:")
            for i, line in enumerate(raw_lines, 1):
                print(f"Line {i}: {line[:200]}...")
        except Exception as e3:
            print(f"Even raw text read failed: {e3}")
        raise

print("\nConnected securely! Your key remains hidden.")
print(f"Schema: {df.schema}")
print(f"Columns: {df.columns}")
print(f"\nNote: Partition columns (year, month, day, hour) are automatically added by Spark")
print(f"Total records: {df.count()}")
display(df.limit(10))